# lora-kernel — S4: the first adapters

**What this measures.** Whether a QLoRA adapter over one base model becomes a
*domain expert* — better than the same base on the same task — and whether an
adapter trained on one clinic beats an adapter trained on another **on its own
region**. The second question is the one the architecture rests on: without
per-region specialisation there is no pool to route between.

**What falsifies it.**
1. The adapter does not beat the base on `val`. Then specialisation did not
   happen and no routing scheme can rescue it.
2. The region experts do not beat each other on their own regions. Then the
   pool is one expert wearing three names.

**The generalisation probe.** `val_delta` is the clinic where one unpublished
rule **inverts**. An adapter that memorised the rule scores well on `val` and
collapses here. That gap is the false-promotion number and it is reported
beside every gain — a gain without it is not a result.

**Hardware.** `gemma-4-E4B-it` trains on a free T4. `gemma-4-26B-A4B-it` needs
an A100 (or an L4 with the short sequences this task has). Set the runtime
before running anything: *Runtime → Change runtime type → GPU*.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install -U transformers peft trl bitsandbytes accelerate datasets

In [ ]:
# The data and the grader travel with the repository, so this notebook is
# reproducible from a clone and nothing is pasted in by hand.
!git clone -q https://github.com/EvolvingAgentsLabs/lora-kernel.git 2>/dev/null || (cd lora-kernel && git pull -q)
%cd lora-kernel
!ls training/data/*.jsonl | head

## Configuration

One base model, changed here and nowhere else. Greedy decoding everywhere,
because every number this project has produced is at temperature 0.

In [ ]:
BASE = "google/gemma-4-E4B-it"      # or "google/gemma-4-26B-A4B-it" on an A100
LOAD_IN_4BIT = True
MAX_SEQ = 1024                      # the canonical prompt is ~250 tokens
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
EPOCHS, LR, BATCH, ACCUM = 3, 2e-4, 4, 4
MAX_NEW_TOKENS = 64                 # the answer is one short JSON object
SEED = 0

import json, torch, random, numpy as np
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
from training.evaluate import load_jsonl, evaluate, compare

VAL       = load_jsonl("training/data/val.jsonl")
VAL_DELTA = load_jsonl("training/data/val_delta.jsonl")
TRAIN     = load_jsonl("training/data/train.jsonl")
print(f"train {len(TRAIN)}  val {len(VAL)}  val_delta {len(VAL_DELTA)}")

## The base model, and its score before anything is trained

The baseline runs first and its number is written down before the adapter
exists. A treatment measured against a baseline recorded afterwards is not a
measurement.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16,
                         bnb_4bit_use_double_quant=True) if LOAD_IN_4BIT else None
tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    BASE, quantization_config=bnb, torch_dtype=torch.bfloat16, device_map="auto")
model.config.use_cache = True

def make_generate(m):
    def gen(system, user):
        msgs = [{"role": "system", "content": system},
                {"role": "user", "content": user}]
        try:
            text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        except Exception:  # a base model without a chat template
            text = f"{system}\n\n{user}\n\n"
        ids = tok(text, return_tensors="pt").to(m.device)
        with torch.no_grad():
            out = m.generate(**ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             pad_token_id=tok.pad_token_id)
        return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    return gen

base_gen = make_generate(model)
base_val   = evaluate(base_gen, VAL, f"base {BASE.split('/')[-1]}")
base_delta = evaluate(base_gen, VAL_DELTA, "base · delta")
print(json.dumps({k: v for k, v in base_val.items() if k != 'records'}, indent=2))
print(json.dumps({k: v for k, v in base_delta.items() if k != 'records'}, indent=2))

## Train the adapter

600 generated cases from the three published clinics. `delta` is deliberately
absent: it is the probe, not training data.

In [ ]:
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

def to_text(rows):
    out = []
    for r in rows:
        try:
            out.append({"text": tok.apply_chat_template(r["messages"], tokenize=False)})
        except Exception:
            m = r["messages"]
            out.append({"text": f"{m[0]['content']}\n\n{m[1]['content']}\n\n{m[2]['content']}"})
    return Dataset.from_list(out)

def train_adapter(rows, out_dir):
    m = prepare_model_for_kbit_training(model) if LOAD_IN_4BIT else model
    peft_model = get_peft_model(m, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"]))
    peft_model.print_trainable_parameters()
    trainer = SFTTrainer(
        model=peft_model, train_dataset=to_text(rows),
        args=SFTConfig(output_dir=out_dir, num_train_epochs=EPOCHS,
                       per_device_train_batch_size=BATCH,
                       gradient_accumulation_steps=ACCUM, learning_rate=LR,
                       max_length=MAX_SEQ, logging_steps=10, seed=SEED,
                       report_to=[], save_strategy="no", bf16=True))
    trainer.train()
    peft_model.save_pretrained(out_dir)
    return peft_model

expert = train_adapter(TRAIN, "adapters/all-clinics")

## The number the step exists for

Adapter minus base, on cases neither of them was trained on — and the delta
probe beside it, because a gain that does not survive the inverted rule is a
memorised rule, not an expert.

In [ ]:
adapter_gen   = make_generate(expert)
adapter_val   = evaluate(adapter_gen, VAL, "adapter")
adapter_delta = evaluate(adapter_gen, VAL_DELTA, "adapter · delta")
print(compare(base_val, adapter_val)); print()
print(compare(base_delta, adapter_delta)); print()
gain  = adapter_val['accuracy']   - base_val['accuracy']
probe = adapter_delta['accuracy'] - base_delta['accuracy']
print(f"gain on the published clinics {gain:+.3f}   ·   on the inverted rule {probe:+.3f}")
if gain > 0 and probe < 0:
    print("FALSE PROMOTION SHAPE: it learned the rule, not the reading. Report both.")

## Two region experts, and the question the pool depends on

`alpha` and `beta` publish different protocols. Train one adapter on each and
cross-evaluate. **Each must be better on its own region than the other is**, or
there is nothing for acceptance routing to choose between and the pool is one
expert wearing three names.

In [ ]:
alpha_rows = [r for r in TRAIN if r['clinic'] == 'alpha']
beta_rows  = [r for r in TRAIN if r['clinic'] == 'beta']
val_alpha  = [r for r in VAL if r['clinic'] == 'alpha']
val_beta   = [r for r in VAL if r['clinic'] == 'beta']

expert_a = train_adapter(alpha_rows, "adapters/alpha")
res = {"alpha_on_alpha": evaluate(make_generate(expert_a), val_alpha, "α on α"),
       "alpha_on_beta":  evaluate(make_generate(expert_a), val_beta,  "α on β")}
expert_b = train_adapter(beta_rows, "adapters/beta")
res["beta_on_beta"]  = evaluate(make_generate(expert_b), val_beta,  "β on β")
res["beta_on_alpha"] = evaluate(make_generate(expert_b), val_alpha, "β on α")

print(f"{'':<10}{'on α':>10}{'on β':>10}")
print(f"{'expert α':<10}{res['alpha_on_alpha']['accuracy']:>10.3f}{res['alpha_on_beta']['accuracy']:>10.3f}")
print(f"{'expert β':<10}{res['beta_on_alpha']['accuracy']:>10.3f}{res['beta_on_beta']['accuracy']:>10.3f}")
own   = res['alpha_on_alpha']['accuracy'] + res['beta_on_beta']['accuracy']
other = res['beta_on_alpha']['accuracy']  + res['alpha_on_beta']['accuracy']
print(f"\nspecialisation: own regions {own/2:.3f} vs other regions {other/2:.3f} "
      f"({(own-other)/2:+.3f})")
if own <= other:
    print("NO SPECIALISATION. The pool has nothing to route between; say so and stop.")

## Save the results and the weights

The JSON goes back into `docs/EXPERIMENT_PLAN.md` §S4; the adapters are what
the next step serves.

In [ ]:
summary = {"base": BASE, "lora": {"r": LORA_R, "alpha": LORA_ALPHA,
           "epochs": EPOCHS, "lr": LR, "train_n": len(TRAIN)},
           "base_val": {k: v for k, v in base_val.items() if k != 'records'},
           "adapter_val": {k: v for k, v in adapter_val.items() if k != 'records'},
           "base_delta": {k: v for k, v in base_delta.items() if k != 'records'},
           "adapter_delta": {k: v for k, v in adapter_delta.items() if k != 'records'},
           "regions": {k: {kk: vv for kk, vv in v.items() if kk != 'records'}
                       for k, v in res.items()}}
open("s4_results.json", "w").write(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2)[:2000])
!zip -qr adapters.zip adapters s4_results.json && echo 'adapters.zip ready to download'